## 3 — Choose SCOPE → clone AOSP + HMI + VSS → index on GPU

`SCOPE`: `automotive` (start here) | `framework` (+frameworks/base) | `full` (bring your own).
Now clones **HMI** (Car UI Library `apps/Car/libs`) and **VSS** (COVESA vehicle_signal_specification,
dropped under `vendor/vss` so the VSS signal-tree chunker + customer priors pick it up).
Clone is non-fatal. Index runs on the free GPU (Ollama not up yet).

Note: the `automotive` scope already includes `vendor/`, so `vendor/vss` (COVESA) gets indexed.

## 0 — GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/android-auto-ai-agent'
STORES=f'{DRIVE}/stores'
os.makedirs(STORES,exist_ok=True)
print('STORES =',STORES)

## 1 — Clone project + deps

In [ ]:
import os
if not os.path.isdir('/content/android-auto-ai-agent'):
    !git clone https://github.com/appdev1307/android-auto-ai-agent.git /content/android-auto-ai-agent
else:
    !cd /content/android-auto-ai-agent && git pull
%cd /content/android-auto-ai-agent
!apt-get -qq install -y ripgrep >/dev/null && echo ripgrep ok
!pip -q install -r requirements.txt
print('project ready')

## 2 — Config: stores→Drive, agent→Ollama

In [ ]:
import yaml, pathlib, os
LLM_MODEL='qwen2.5-coder:32b'
API_BASE='http://127.0.0.1:11434/v1'          # Ollama OpenAI-compatible
cfg_path=pathlib.Path('data/config.yaml'); cfg=yaml.safe_load(cfg_path.read_text())
cfg['model']['name']=LLM_MODEL
cfg['model']['api_base']=API_BASE
cfg['rag']['stores_root']=STORES
cfg_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
os.environ['OPENAI_API_KEY']='ollama'          # dummy; ignored
print('LLM   =',LLM_MODEL,'@',API_BASE)
print('embed =',cfg['rag']['embed_model'])

## 3 — Choose SCOPE → clone AOSP → index on GPU

`SCOPE`: `automotive` (start here) | `framework` (+frameworks/base, big) | `full` (bring your own).
Clone is non-fatal (a repo that fails is skipped). Index runs on the free GPU (Ollama not up yet).

In [ ]:
SCOPE = 'automotive'
TAG   = 'android-15.0.0_r1'
import os, subprocess, pathlib
AOSP='/content/aosp'
# AOSP layers (aosp-mirror GitHub, layout the indexer expects)
MIRROR={
  'hardware/interfaces':        'https://github.com/aosp-mirror/platform_hardware_interfaces',
  'packages/services/Car':      'https://github.com/aosp-mirror/platform_packages_services_car',
  'packages/apps/Car/Settings': 'https://github.com/aosp-mirror/platform_packages_apps_Car_Settings',
  'packages/apps/Car/libs':     'https://github.com/aosp-mirror/platform_packages_apps_Car_libs',      # HMI: Car UI Library
  'frameworks/base':            'https://github.com/aosp-mirror/platform_frameworks_base',
}
SCOPE_REPOS={
  'automotive':['hardware/interfaces','packages/services/Car','packages/apps/Car/Settings','packages/apps/Car/libs'],
  'framework' :['hardware/interfaces','packages/services/Car','packages/apps/Car/Settings','packages/apps/Car/libs','frameworks/base'],
  'full'      :[],
}
def clone(rel):
    dest=f'{AOSP}/{rel}'
    if os.path.isdir(dest): print('  have',rel); return
    url=MIRROR[rel]; os.makedirs(pathlib.Path(dest).parent,exist_ok=True)
    for args in (['--depth=1','-b',TAG,url,dest], ['--depth=1',url,dest]):
        if subprocess.run(['git','clone',*args]).returncode==0: print('  cloned',rel); return
    print('  SKIP (clone failed):',rel)
for rel in SCOPE_REPOS.get(SCOPE,[]): clone(rel)

# VSS from COVESA (NOT AOSP layout) -> drop under vendor/vss so the customer/vss
# path priors + VSS signal-tree chunker pick it up. Non-fatal.
VSS_DEST=f'{AOSP}/vendor/vss'
if not os.path.isdir(VSS_DEST):
    os.makedirs(pathlib.Path(VSS_DEST).parent, exist_ok=True)
    r=subprocess.run(['git','clone','--depth=1',
                      'https://github.com/COVESA/vehicle_signal_specification', VSS_DEST])
    print('  cloned VSS (COVESA)' if r.returncode==0 else '  SKIP VSS (clone failed)')
else:
    print('  have vendor/vss')

os.environ['AOSP_ROOT']=AOSP
print('SCOPE =',SCOPE,'| tree at',AOSP)
!du -sh {AOSP} 2>/dev/null

In [ ]:
import os
if os.path.exists(f'{STORES}/_base/aosp15/manifest.json'):
    print('index already on Drive -> skip (delete manifest to rebuild / change scope)')
else:
    !python -m retrieval.indexer --aosp-root {AOSP} --base --aosp-version aosp15 --scope {SCOPE}
print('index at', f'{STORES}/_base/aosp15')

## 4 — Start Ollama + pull model
Ollama = prebuilt, **no CUDA/torch compile**. `ensure_llm()` is idempotent — re-run anytime.

In [ ]:
import os, subprocess, time, requests
OLLAMA_URL='http://127.0.0.1:11434'
def _alive():
    try: return requests.get(OLLAMA_URL+'/api/tags',timeout=2).ok
    except Exception: return False
def ensure_llm():
    if not _alive():
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True)
        subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True, env={**os.environ})
        for i in range(30):
            if _alive(): break
            time.sleep(2)
    # pull is a no-op if already present
    print('pulling qwen2.5-coder:32b (first time ~20GB, a few minutes)...')
    subprocess.run('ollama pull qwen2.5-coder:32b', shell=True)
    print('ollama up' if _alive() else 'ollama DOWN')
    return OLLAMA_URL+'/v1'
ensure_llm()

## 5 — Run agent
Embedder on CPU (Ollama holds the GPU); query embedding is a few cheap calls.
Includes the per-layer specialist pass.

In [ ]:
BUG="Android 15: VSS Vehicle.Speed not updating in HMI after ignition ON"
!CUDA_VISIBLE_DEVICES="" python -m agent.main --bug "{BUG}" --aosp-root /content/aosp --aosp-version aosp15

---
### Notes
- **No compile anywhere** — Ollama ships prebuilt binaries; avoids the vLLM/torch/flashinfer
  version+JIT-compile issues on Colab Python 3.13.
- **New session** → cells 0,1,2,3 (index skips, reused from Drive), 4 (ollama), 5.
- **Ollama died** → re-run cell 4 (`ensure_llm()`); restarts only if needed.
- **Change coverage** → set SCOPE in cell 3, delete `stores/_base/aosp15/manifest.json`, re-run cell 3.
- **Lighter model if 32B is slow/tight** → set LLM_MODEL='qwen2.5-coder:14b' in cell 2 and
  `ollama pull qwen2.5-coder:14b` in cell 4.